# GPT-2 / Llama 3.1 8B Seinfeld Finetuning (LoRA / QLoRA)

Fine-tunes a causal LM on Seinfeld scripts using **LoRA** (or **QLoRA** for large models).

## Models supported
| `BASE_MODEL` | VRAM needed | Colab tier | Notes |
|---|---|---|---|
| `gpt2-medium` | ~4 GB | Free T4 | LoRA fp16, ~15 min |
| `meta-llama/Meta-Llama-3.1-8B` | ~24 GB | A100 (40/80 GB) | QLoRA 4-bit, ~45 min |

## What you need
- **GPT-2**: any Colab GPU (free T4 is fine)
- **Llama 3.1 8B**: Colab A100 (paid) + HuggingFace account with Llama access granted at huggingface.co/meta-llama/Meta-Llama-3.1-8B + `HF_TOKEN` in Colab Secrets

## Before you start
**Runtime → Change runtime type → GPU** (T4 for GPT-2, A100 for Llama)

In [ ]:
# 1. Clone repo
!git clone https://github.com/detrin/gpt-seinfeld /content/gpt-seinfeld
%cd /content/gpt-seinfeld

In [ ]:
# Install dependencies
# bitsandbytes is only needed for Llama (QLoRA 4-bit), harmless to install for GPT-2
!pip install -q "transformers>=4.44" "peft>=0.17.0" "bitsandbytes>=0.43.0" datasets accelerate torch

In [ ]:
# HuggingFace login — required for Llama 3.1 (gated model), optional for GPT-2
import os

from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")  # set in Colab Secrets (key icon in left sidebar)
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login

    login(token=hf_token)
    print("HF login OK")
else:
    print("No HF_TOKEN found — fine for GPT-2, required for Llama")

In [ ]:
# Mount Google Drive — checkpoint saved here so it survives session resets
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!nvidia-smi | head -5

In [ ]:
import os

os.chdir("/content/gpt-seinfeld")

# ── Choose your model ──────────────────────────────────────────────────────────
# GPT-2 medium (free T4, ~15 min):
os.environ["BASE_MODEL"] = "gpt2-medium"
os.environ["CHECKPOINT_DIR"] = "/content/drive/MyDrive/gpt2-seinfeld-v2"

# Llama 3.1 8B (A100, ~45 min) — uncomment to use:
# os.environ["BASE_MODEL"]      = "meta-llama/Meta-Llama-3.1-8B"
# os.environ["CHECKPOINT_DIR"]  = "/content/drive/MyDrive/llama-seinfeld"

os.environ["DATA_FILE"] = "data/processed/train.jsonl"

%run training/train.py

In [ ]:
# Inline inference test — runs directly from the merged checkpoint, no export needed
import torch
from transformers import pipeline

CHECKPOINT = os.environ.get("CHECKPOINT_DIR", "/content/drive/MyDrive/gpt2-seinfeld-v2")
device = 0 if torch.cuda.is_available() else -1

gen = pipeline("text-generation", model=CHECKPOINT, device=device)

topics = ["losing a parking spot", "a bad haircut", "waiting for a table at a restaurant"]
for topic in topics:
    out = gen(
        f"TOPIC: {topic}\n\n[",
        max_new_tokens=250,
        do_sample=True,
        temperature=0.9,
        repetition_penalty=1.2,
    )
    text = out[0]["generated_text"]
    start = text.index("\n\n[") + 3
    print(f"{'=' * 60}\nTOPIC: {topic}\n{'=' * 60}")
    print("[" + text[start:])
    print()